# AAT-2 · Information-Retrieval Question Answering
Nideesh V · 1GA23AI028

In [1]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
documents = open('../data/documents.txt').read().strip().split('\n')
for i, d in enumerate(documents, 1):
    print(f'D{i}:', d)

D1: The Taj Mahal stands in Agra on the bank of the Yamuna river. It was commissioned by the Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal.
D2: Hampi was the capital of the Vijayanagara Empire. Its ruins in Karnataka are a UNESCO World Heritage Site.
D3: The Gateway of India in Mumbai was built to commemorate the visit of King George V and Queen Mary. The monument was completed in 1924.
D4: Kaziranga National Park in Assam is famous for the one-horned rhinoceros. The park is also a tiger reserve.
D5: The Konark Sun Temple in Odisha was built in the 13th century. It is designed in the shape of a giant chariot with carved stone wheels.


In [2]:
sentences, owner = [], []
for d, text in enumerate(documents, 1):
    for s in nltk.sent_tokenize(text):
        sentences.append(s); owner.append(d)
vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), sublinear_tf=True)
matrix = vec.fit_transform(sentences)
print('Sentences indexed:', len(sentences), '| TF-IDF matrix:', matrix.shape)

Sentences indexed: 10 | TF-IDF matrix: (10, 113)


In [3]:
def answer(question, k=3):
    scores = cosine_similarity(vec.transform([question]), matrix)[0]
    order = scores.argsort()[::-1][:k]
    print('Q:', question)
    for r, i in enumerate(order, 1):
        print(f'  {r}. [D{owner[i]}] {scores[i]:.4f}  {sentences[i]}')
    best = order[0]
    window = [j for j in (best - 1, best, best + 1) if 0 <= j < len(sentences) and owner[j] == owner[best] and (j == best or scores[j] > 0)]
    print('  TOP SENTENCE  ->', sentences[best])
    print('  ANSWER WINDOW ->', ' '.join(sentences[j] for j in window), '\n')

In [4]:
for q in ['Who commissioned the Taj Mahal?', 'Which national park is known for the one-horned rhinoceros?',
          'When was the Gateway of India completed?', 'Which empire had Hampi as its capital?']:
    answer(q)

Q: Who commissioned the Taj Mahal?
  1. [D1] 0.3956  The Taj Mahal stands in Agra on the bank of the Yamuna river.
  2. [D1] 0.2183  It was commissioned by the Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal.
  3. [D5] 0.0000  The Konark Sun Temple in Odisha was built in the 13th century.
  TOP SENTENCE  -> The Taj Mahal stands in Agra on the bank of the Yamuna river.
  ANSWER WINDOW -> The Taj Mahal stands in Agra on the bank of the Yamuna river. It was commissioned by the Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal. 

Q: Which national park is known for the one-horned rhinoceros?
  1. [D4] 0.6707  Kaziranga National Park in Assam is famous for the one-horned rhinoceros.
  2. [D4] 0.1390  The park is also a tiger reserve.
  3. [D5] 0.0000  The Konark Sun Temple in Odisha was built in the 13th century.
  TOP SENTENCE  -> Kaziranga National Park in Assam is famous for the one-horned rhinoceros.
  ANSWER WINDOW -> Kaziranga National Park in Assam is famous f

In [5]:
answer('What is the capital of France?')  # out-of-collection question

Q: What is the capital of France?
  1. [D2] 0.3780  Hampi was the capital of the Vijayanagara Empire.
  2. [D5] 0.0000  It is designed in the shape of a giant chariot with carved stone wheels.
  3. [D5] 0.0000  The Konark Sun Temple in Odisha was built in the 13th century.
  TOP SENTENCE  -> Hampi was the capital of the Vijayanagara Empire.
  ANSWER WINDOW -> Hampi was the capital of the Vijayanagara Empire. 

